# Churn Model — Logistic Regression on `ml_customer_features`

Trains a baseline binary classifier on the customer-level features produced by `ml_features.ipynb`. Output: a fitted `PipelineModel` and a held-out AUC.

**Pipeline:**
1. Load `ml_customer_features` via `spark.table()`.
2. 80/20 train/test split on the raw feature table (BEFORE assembly, to prevent test-set leakage into any future fitted preprocessor).
3. `VectorAssembler` → `LogisticRegression`, wrapped in a `Pipeline` so the same preprocessing is applied identically at inference time.
4. Evaluate on the test set with `BinaryClassificationEvaluator` (AUC-ROC).

**Design choices:**

- **Numeric features only for the baseline.** `loyalty_tier` and `primary_region` are categorical — they'd need `StringIndexer` + `OneHotEncoder` to enter a linear model, which doubles the pipeline length. Worth adding once the numeric-only baseline is validated.
- **`customer_id` is excluded.** It's an identifier, not a feature; including it would let the model memorize the training set.
- **Date columns excluded.** Their information is already encoded in `customer_tenure_days` / `days_since_last_purchase`. Keeping both injects collinearity.
- **`Pipeline` over `assembler.transform() → lr.fit()`.** A fitted `PipelineModel` carries every stage's state, so inference code is one `.transform()` call regardless of how many preprocessing stages we add later.
- **Cache the train DataFrame.** L-BFGS makes ~10-20 passes during LR fitting; caching pays for itself on the first iteration.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder.getOrCreate()

## 1. Load `ml_customer_features`

Read the persisted feature table written by `ml_features.ipynb`. We filter out any rows with a NULL label — defensive against future feature-set changes that could introduce them.

In [0]:
ml_data = (
    spark.table("ml_customer_features")
    .filter(F.col("churn_label").isNotNull())
)

# Schema check — confirms the columns we plan to use exist with expected types.
ml_data.printSchema()

## 2. Train/test split

Split BEFORE feature assembly. If we assembled+fit a scaler on the full table and then split, statistics from the test set would leak into the model. With raw splits, any future scaler fits only on `train_df`.

`randomSplit` is Spark's per-row Bernoulli sampler. Seed makes the split reproducible across notebook reruns — essential when comparing model variants.

**No DataFrame cache** — Databricks serverless raises `[NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE` on `.cache()` / `.persist()`. Instead we rely on the Delta storage cache + Photon, which transparently serve repeat reads of the same Parquet files from local SSD. L-BFGS re-scans `train_df` on each iteration, but those scans hit warm files after the first pass.

In [0]:
train_df, test_df = ml_data.randomSplit([0.8, 0.2], seed=42)

# Note: no .cache() / .persist(). Databricks serverless rejects PERSIST TABLE; rely on
# Photon + the Delta storage cache, which kick in transparently for repeat reads of the
# same files. L-BFGS will re-scan train_df on each iteration but those scans are fast
# against cached Parquet.

# count() confirms the 80/20 split landed as expected.
print(f"Train rows: {train_df.count():,}")
print(f"Test rows:  {test_df.count():,}")

# Class balance — important context for interpreting AUC. If churn_label is 99% one class,
# even a constant predictor scores ~0.5 AUC, but accuracy would be misleadingly high.
print("\nTraining class balance:")
train_df.groupBy("churn_label").agg(F.count("*").alias("n")).orderBy("churn_label").show()


## 3. Assemble + train

Spark ML's training contract: the estimator wants a single `Vector`-typed column (named `features` by default). `VectorAssembler` packs N scalar columns into that one column — no statistical work, just shape transformation.

**Feature selection rationale (numeric-only baseline):**
- `loyalty_tier` / `primary_region` are categorical — would need `StringIndexer` + `OneHotEncoder`. Skip for the baseline.
- `customer_id` is an identifier. Excluded.
- `first_purchase_date` / `last_purchase_date` excluded; same info in `customer_tenure_days` / `days_since_last_purchase`.
- `churn_label` is the target — never include it in features (would be perfect leakage).

In [0]:
# Numeric feature list. Maintain this list explicitly rather than "all numeric columns":
# explicit naming catches accidental inclusion of a leaky column when the source schema grows.
feature_cols = [
    "total_gross_revenue",
    "total_discount",
    "total_revenue",
    "total_cost",
    "total_gross_margin",
    "total_quantity",
    "transaction_count",
    "promoted_transaction_count",
    "avg_discount_pct",
    "distinct_categories",
    "distinct_brands",
    "distinct_stores",
    "distinct_promotion_types",
    "avg_transaction_value",
    "gross_margin_pct",
    "promo_transaction_pct",
    "customer_tenure_days",
    "days_since_last_purchase",
]

# handleInvalid="skip" drops rows with any NULL feature value. Alternative: "keep" emits NaN
# vectors which LR can't handle, or "error" which fails the job. "skip" is the safe default
# for a baseline; in production an upstream Imputer stage would be cleaner.
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip",
)

# LR config:
#   - labelCol/featuresCol: standard Spark ML wiring.
#   - maxIter=20: L-BFGS usually converges well before 20 on this problem size; if it
#     hits the cap the training log will warn.
#   - regParam=0.0: no regularization for the baseline. Add elasticNet/L2 once we have a
#     baseline AUC to compare against.
#   - standardization=True (default): LR will standardize features internally for the
#     L-BFGS solver, then un-standardize the coefficients. This is why we don't need a
#     separate StandardScaler in the pipeline.
lr = LogisticRegression(
    labelCol="churn_label",
    featuresCol="features",
    maxIter=20,
    regParam=0.0,
)

# Pipeline: assembler -> lr. Fit once, transform anywhere — both stages are reused at
# inference time via PipelineModel.transform().
pipeline = Pipeline(stages=[assembler, lr])

# Fit. This runs the assembler (deterministic, single pass) then the LR (multiple passes
# for L-BFGS). Total is ~maxIter scans of train_df; on serverless these hit the Delta
# storage cache after the first pass, so per-iteration cost stays low.
model = pipeline.fit(train_df)

# Quick training diagnostic: did L-BFGS actually converge, or did we hit maxIter?
lr_model = model.stages[-1]
training_summary = lr_model.summary
print(f"Iterations run: {training_summary.totalIterations} (cap: {lr.getMaxIter()})")
print(f"Final objective (training): {training_summary.objectiveHistory[-1]:.6f}")
print(f"Training AUC: {training_summary.areaUnderROC:.4f}")

## 4. Evaluate on the held-out test set

AUC-ROC is the right primary metric for this problem:
- **Threshold-independent.** Unlike accuracy/precision/recall, it doesn't depend on where we set the decision boundary. For churn we'll usually set the threshold by business cost (cost of intervention vs cost of letting a customer churn), not by 0.5.
- **Robust to class imbalance.** If churn rate is 5%, a constant-zero classifier has 95% accuracy but 0.5 AUC. The latter is the honest signal.

**Sanity-check coefficients** afterward: a baseline LR should show negative coefficients on `total_revenue`, `transaction_count`, `customer_tenure_days` (more engagement → less churn) and a positive coefficient on `days_since_last_purchase` (the label is *defined* from this column, so the relationship has to be strong and positive). If signs come out wrong, the model has a bug — not a modeling problem.

In [0]:
# Apply the FULL pipeline (assembler + LR) to the test set. PipelineModel.transform()
# runs every stage, so test predictions go through identical preprocessing as training.
predictions = model.transform(test_df)

# BinaryClassificationEvaluator with rawPredictionCol="rawPrediction":
# LogisticRegression emits both "rawPrediction" (logits) and "probability" (sigmoid'd).
# AUC is computed from rawPrediction because monotonic transforms don't change the ROC.
evaluator = BinaryClassificationEvaluator(
    labelCol="churn_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
test_auc = evaluator.evaluate(predictions)
print(f"Test AUC-ROC: {test_auc:.4f}")

# Also compute PR AUC. For imbalanced classes (which churn often is), PR AUC is more
# discriminating than ROC AUC at the high-precision end of the curve.
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="churn_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
)
test_pr_auc = pr_evaluator.evaluate(predictions)
print(f"Test AUC-PR:  {test_pr_auc:.4f}")

# Coefficient sanity check. Pair feature names with their fitted coefficients so we can
# read direction-of-effect at a glance. Done as a small Python list because there are
# only ~20 coefficients — driver-side work is fine here.
print("\nFeature coefficients (sorted by absolute value):")
coeffs = list(zip(feature_cols, lr_model.coefficients.toArray()))
for name, c in sorted(coeffs, key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name:<30s} {c:+.4f}")
print(f"  {'(intercept)':<30s} {lr_model.intercept:+.4f}")

# Sample predictions for visual sanity check.
print("\nSample test predictions:")
predictions.select(
    "customer_id",
    "churn_label",
    "prediction",
    F.round(F.col("probability").getItem(1), 4).alias("churn_probability"),
).show(10, truncate=False)